**This script processes multiple prompts and data files in batch using APIs**

1. Model Initialization: sets up API fucntion calls once and reuses it for all processing

2. List Management:
Reads prompt filenames from prompt_list.txt
Reads data filenames from data_list.txt


3. Batch Processing:
Iterates through each prompt file
For each prompt, processes it with each data file
Generates responses for all prompt-data combinations

4. Outputs:
Saves outputs using the prompt filename as base
Adds timestamps to avoid overwriting
Organizes all outputs in an outputs/ directory

In [ ]:
#pip install OpenAI

In [ ]:
#pip install anthropic

In [ ]:
#pip install instructor

In [ ]:
#pip install python-docx

In [4]:
# Import std libraries
import os
import pandas as pd
from pathlib import Path
import json
from datetime import datetime

# Import Word doc libaries
from docx import Document
from docx.shared import Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_ALIGN_VERTICAL


In [ ]:
# Set up API keys in env vars
os.environ["GEMINI_API_KEY"] = ""
os.environ["OPENAI_API_KEY"] = ""
os.environ["ANTHROPIC_API_KEY"] = ""
os.environ["GROK_API_KEY"] = ""


In [6]:
def initialize_model(model_name):
    # Set up function for models.

    print(f"Initializing model: {model_name}")
    print("-" * 50)

    try:
      if model_name == "gpt":
        from openai import OpenAI
        import instructor
        model = instructor.patch(OpenAI(api_key=os.environ["OPENAI_API_KEY"]))

      elif model_name == "claude":
        from anthropic import Anthropic
        model = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

      elif model_name == "gemini":
        import google.generativeai as genai
        model = genai.GenerativeModel("gemini-3-pro-preview")

      elif model_name == "grok":
        from openai import OpenAI
        # Initialize the OpenAI client, pointing it to the Grok API endpoint
        model = OpenAI(
          api_key=os.environ["GROK_API_KEY"],
          base_url="https://api.x.ai/v1", # Grok API base URL
          )

      print(f"✓ Model '{model_name}' loaded successfully!")
      print("-" * 50)

      return model

    except ModelNotFoundError:
      print(f"✗ Error: {model_name} not found!")
      return []

In [ ]:
def load_file_list(list_file):
    # Load list of filenames from a text file.
    try:
        with open(list_file, 'r', encoding='windows-1252') as f:
            # Read lines and remove empty lines and whitespace
            filenames = [line.strip() for line in f.readlines() if line.strip()]
        print(f"✓ Loaded {len(filenames)} filenames from {list_file}")
        return filenames
    except FileNotFoundError:
        print(f"✗ Error: {list_file} not found!")
        return []

In [ ]:
def load_prompt_file(prompt_file):
    # Load list of prompt filenames from a text file.
    try:
        with open(prompt_file, 'r', encoding='windows-1252') as f:
            prompt = f.read().strip()
        return prompt
    except FileNotFoundError:
        print(f"  ✗ Warning: Prompt file '{prompt_file}' not found. Using default prompt.")
        return "Please analyze the following data and provide insights:"

In [9]:
def load_data_file(data_file):
    # Load data from a text file.
    try:
        with open(data_file, 'r', encoding='windows-1252') as f:
            data = f.read().strip()
        return data
    except FileNotFoundError:
        print(f"  ✗ Warning: Data file '{data_file}' not found. Using empty data.")
        return ""

In [10]:
def generate_response(model_name, model, prompt, data, max_length, temp):
    # Generate response from the model for given prompt and data.

    # Combine prompt and data
    full_input = f"Prompt:\n{prompt}\n\nData:\n{data}\n\nResponse:"

    #print(f"Full Input:\n{full_input}")

    generated_text = ""

    # Make the API call to generate content
    if model_name=="gpt":
      response_obj = model.chat.completions.create(
        model="gpt-5.2",
        messages=[
          {"role": "system", "content": prompt},
          {"role": "user", "content": data},
        ],
        temperature=temp,
      )
      generated_text = response_obj.choices[0].message.content

    elif model_name=="claude":
      response_obj = model.messages.create(
        model="claude-opus-4-5-20251101",
        max_tokens=max_length,
        system=prompt,
        messages=[
            {"role": "user", "content": [
                {
                    "type": "text",
                    "text": full_input
                }
            ]}
        ],
        temperature=temp,
      )
      generated_text = response_obj.content[0].text

    elif model_name=="gemini":
        response_obj = model.generate_content(
          full_input,
          generation_config={"temperature": temp, "max_output_tokens": max_length}
        )
        generated_text = response_obj.text

    elif model_name=="grok":
      response_obj = model.chat.completions.create(
        model="grok-4",
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": data},
        ],
        temperature=temp,
        )
      generated_text = response_obj.choices[0].message.content

    return generated_text

In [11]:
def save_response(response, prompt_name, data_file, model_name, temp, output_dir="outputs"):
    # Save the model response to a text file using the prompt name.

    # Create output directory if it doesn't exist
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    # Generate output filename based on prompt name
    base_name = Path(prompt_name).stem  # Remove extension
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_file = Path(output_dir) / f"{model_name}_{temp}_{base_name}_response.txt"

    # Save response
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(f"Model: {model_name}\n")
        f.write(f"Temp: {temp}\n")
        f.write(f"Prompt File: {prompt_name}\n")
        f.write(f"Data File: {data_file}\n")
        f.write(f"Generated at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write("-" * 50 + "\n\n")
        f.write(response)

    print(f"  ✓ Response saved to: {output_file}")
    return output_file

In [20]:
def replace_text_in_document(document, tag, new_text):
    # Remove extension
    base_tag = Path(tag).stem

    # Add $ sign to beginining of tag
    TAG = f"${base_tag}"

    # Replaces TAG with new_text in the document's paragraphs.
    for paragraph in document.paragraphs:
        if TAG in paragraph.text:
            paragraph.text = paragraph.text.replace(TAG, new_text)


In [24]:
def insert_table_in_document(document, tag, new_text):
    # Inserts a markdown-formatted table into a Word document at the location of a specified tag.

    # Remove extension
    base_tag = Path(tag).stem

    # Add $ sign to beginining of tag
    TAG = f"${base_tag}"

    # Find the paragraph containing the tag
    target_paragraph = None
    for paragraph in document.paragraphs:
        if TAG in paragraph.text:
            target_paragraph = paragraph
            break

    if not target_paragraph:
        print(f"TAG '{TAG}' not found in the document.")
        return

    # Parse the markdown table from new_text
    # Rows are delimited by '\n' (newline character)
    rows_str = new_text.strip().split('\n')
    rows_str = [row.strip() for row in rows_str if row.strip()]

    if len(rows_str) < 3: # Need at least header, separator, and one data row
        print("Insufficient table data found in new_text. Expected at least header, separator, and one data row.")
        return

    # Extract header line
    header_line = rows_str[0]
    headers = [h.strip() for h in header_line.strip('|').split('|') if h.strip()]

    # Data lines (skip header and separator)
    data_rows_raw = rows_str[2:]

    table_data = []
    for row_str in data_rows_raw:
        row_data = [d.strip() for d in row_str.strip('|').split('|') if d.strip()]
        if row_data:
            table_data.append(row_data)

    num_cols = len(headers)
    num_rows = len(table_data) + 1 # +1 for header row

    if num_cols == 0:
        print("No columns found in table data.")
        return

    # Create the table
    doc_table = document.add_table(rows=num_rows, cols=num_cols)
    doc_table.style = 'Table Grid' # Apply a style for better visual representation

    # Populate the table
    # Add headers
    hdr_cells = doc_table.rows[0].cells
    for i, header_text in enumerate(headers):
        hdr_cells[i].text = header_text
        hdr_cells[i].paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER
        hdr_cells[i].vertical_alignment = WD_ALIGN_VERTICAL.CENTER

    # Add data rows
    for r_idx, row_data in enumerate(table_data):
        row_cells = doc_table.rows[r_idx + 1].cells # +1 to skip header row
        for c_idx, cell_text in enumerate(row_data):
            if c_idx < num_cols: # Ensure we don't go out of bounds
                row_cells[c_idx].text = cell_text
            row_cells[c_idx].vertical_alignment = WD_ALIGN_VERTICAL.TOP

    # Move the table element to the location of the target paragraph
    parent_element = target_paragraph._element.getparent()
    table_element = doc_table._element
    idx = parent_element.index(target_paragraph._element)

    # Remove the table from its current position (end of document body)
    table_element.getparent().remove(table_element)

    # Insert the table element directly before the target paragraph's element
    parent_element.insert(idx, table_element)

    # Delete the original paragraph containing the tag
    p = target_paragraph._element
    p.getparent().remove(p)


In [26]:
def process_batch(model_name, model, prompt_files, data_files, max_length, temp, template):
    # Process all combinations of prompts and data files.

    output_files = []
    num_pairs = min(len(prompt_files), len(data_files))
    current = 0

    tables_list = ['PromptRegionPlan.txt', 'PromptDistrictPlan.txt', 'PromptLocalPlans.txt', 'Prompt9-1Directions.txt', 'PromptSEPP.txt']

    print(f"\nProcessing prompt pairs for {model_name}...")
    print("=" * 50)

    # Process this single pair
    for i in range(num_pairs):
        prompt_file = prompt_files[i]   # Get prompt at index i
        data_file = data_files[i]       # Get data at same index i

        # Load prompt once for all data files
        prompt_text = load_prompt_file(prompt_file)

        # Load data
        data_text = load_data_file(data_file)

        #print(f"\nPrompt: {prompt_file}")
        #print(f"\nData: {data_file}")

        try:
            # Generate response
            print(f"    Generating response for {model_name} with temperature {temp}")
            response = generate_response(model_name, model, prompt_text, data_text, max_length, temp)

            # Save response with prompt name
            output_file = save_response(response, prompt_file, data_file, model_name, temp)
            output_files.append(output_file)

            # Replace the TAG in DOCX with the LLM reponse
            if prompt_file in tables_list:
              insert_table_in_document(template, prompt_file, response)
            else:
              replace_text_in_document(template, prompt_file, response)

        except Exception as e:
            print(f"    ✗ Error processing combination: {str(e)}")
            continue

    return output_files

In [ ]:
# Main function

# Set model to generate report
MODEL_NAME="gpt"
#MODEL_NAME="claude"
#MODEL_NAME="grok"
#MODEL_NAME="gemini"

# Set Temperature for model
TEMP=0.2
# Set Planning Proposal number for run
PP_NUM='PP-2025-356'
# Set test run number
TEST_NUM='1'

PROMPT_LIST_FILE="prompt_list.txt"
DATA_LIST_FILE="data_list.txt"
OUTPUT_DIR="outputs"
MAX_LENGTH=2048  # Max tokens to generate (about 1500 words)

print("\n" + "=" * 50)
print("BATCH PROCESSING PIPELINE")
print("=" * 50)

try:
    # Step 1: Initialize model
    model = initialize_model(MODEL_NAME)

    # Step 2: Open Docx template
    template_document = Document('TEST_GD_Report_Template.docx')
    print("\n" + "=" * 50)
    print("TEMPLATE DOCX OPENED")
    print("=" * 50)

    # Step 3: Load lists of prompt and data filenames
    print("\n" + "=" * 50)
    print("LOADING PROMPT AND DATA FILES")
    print("=" * 50)
    prompt_files = load_file_list(PROMPT_LIST_FILE)
    data_files = load_file_list(DATA_LIST_FILE)

    if not prompt_files:
        print(f"\n✗ No prompt files found in {PROMPT_LIST_FILE}")
        print("Please create this file with one prompt filename per line.")
        #return

    if not data_files:
        print(f"\n✗ No data files found in {DATA_LIST_FILE}")
        print("Please create this file with one data filename per line.")
        #return

    print(f"\nFound {len(prompt_files)} prompts and {len(data_files)} data files")

    # Step 4: Process all combinations
    output_files = process_batch(
        MODEL_NAME,
        model,
        prompt_files,
        data_files,
        max_length=MAX_LENGTH,
        temp=TEMP,
        template=template_document
    )

    # Step 5: Save the modified document to a new file to avoid overwriting the original template
    docx_filename = f"TEST_{TEST_NUM}_{MODEL_NAME}_Temp={TEMP} ({PP_NUM}) - GD_Report.docx"
    template_document.save(docx_filename)
    print("\n" + "=" * 50)
    print("TEMPLATE DOCX SAVED")
    print("=" * 50)
    #print(f"Tag '{Tag}' replaced with '{input_string}' and saved to '{output_filename}'")

    if output_files:
        print("\n" + "=" * 50)
        print(f"✓ PROCESSING COMPLETE!")
        print(f"✓ Generated {len(output_files)} response files in '{OUTPUT_DIR}' directory")
    else:
        print("\n✗ No files were successfully processed.")

except Exception as e:
    print(f"\n✗ Critical error: {str(e)}")
    import traceback
    traceback.print_exc()


BATCH PROCESSING PIPELINE
Initializing model: gpt
--------------------------------------------------
✓ Model 'gpt' loaded successfully!
--------------------------------------------------

TEMPLATE DOCX OPENED

LOADING PROMPT AND DATA FILES
✓ Loaded 14 filenames from prompt_list.txt
✓ Loaded 14 filenames from data_list.txt

Found 14 prompts and 14 data files

Processing prompt pairs for gpt...
    Generating response for gpt with temperature 0.7
  ✓ Response saved to: outputs/gpt_0.7_PromptObjectivesofPlanningProposal_response.txt
    Generating response for gpt with temperature 0.7
  ✓ Response saved to: outputs/gpt_0.7_PromptExplanationofProvisions_response.txt
    Generating response for gpt with temperature 0.7
  ✓ Response saved to: outputs/gpt_0.7_PromptSiteDescription_response.txt
    Generating response for gpt with temperature 0.7
  ✓ Response saved to: outputs/gpt_0.7_PromptBackground_response.txt
    Generating response for gpt with temperature 0.7
  ✓ Response saved to: out